# LoRA Scan Sweep on Google Colab

**Important:** Make sure to select a GPU runtime:
1. Runtime → Change runtime type
2. Select GPU (T4, L4, or A100)
3. Click Save

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone the repository
!git clone https://github.com/ubitquitin/lora_scan.git
%cd lora_scan

In [ ]:
# Install micromamba (lightweight conda alternative)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba shell init -s bash -p ~/micromamba

In [ ]:
# Create environment and install dependencies
!./bin/micromamba create -n spar python=3.12 -y -c conda-forge
!./bin/micromamba run -n spar pip install -e .

In [ ]:
# Run a test to verify setup
!./bin/micromamba run -n spar python scripts/run.py configs/experiments/sft_sweep_mmlu_gsm8k.yaml --dry-run | head -20

In [ ]:
# Run the full sweep (this will take a while!)
# The sweep will automatically save progress and can be resumed if interrupted
!./bin/micromamba run -n spar python scripts/run.py configs/experiments/sft_sweep_mmlu_gsm8k.yaml

In [ ]:
# Download results
# After the sweep completes, download the results
!tar -czf results.tar.gz runs/
from google.colab import files
files.download('results.tar.gz')

## Resume after interruption

If Colab disconnects or you need to stop, just re-run the sweep command:

```python
!./bin/micromamba run -n spar python scripts/run.py configs/experiments/sft_sweep_mmlu_gsm8k.yaml
```

It will skip completed runs and resume from where it left off!

## Analyze results

After the sweep completes:

In [ ]:
# Load and display results
import sys
sys.path.insert(0, 'src')

from lorascan import runs
import pandas as pd

results = runs.collect_results()
df = pd.DataFrame(results)
print(f"\nCompleted runs: {len(df)}")
print(df.head())

# Summary statistics
print("\n=== Results Summary ===")
print(df.groupby(['model', 'dataset', 'prompt'])['accuracy'].agg(['mean', 'std', 'count']))